# Task A Run 35 -- holdout check: exact twin of the TAPT submission (Task A Run 33)

Task A Run 33, the TAPT submission, does three things:

1. TAPT on the comments' text plus the external Kannada corpus.
2. Four Gemma-4-12B classifiers (seeds 42-45, 3 epochs) continue from that adapter.
3. Their probabilities are averaged.

**This notebook runs exactly that recipe, on the 85% split.** TAPT reads only the 6,114
training comments plus the external corpus, and the four classifiers train on the same
6,114 rows. Everything is scored on the 1,079-row holdout. The one difference from Run 33
is that holdout rows are kept out of everything.

**Two numbers come out of it:**

| number | groups compared | what it tells you |
|---|---|---|
| the verdict, a fair test of TAPT | TAPT at 2 seeds vs Run 28's recipe (no TAPT) at the same 2 seeds | whether TAPT helps, with seed count held equal |
| the expected submission level | TAPT at 4 seeds, exactly what Run 33 submits | what to expect from the submission |

Run 28's seed-42 holdout probabilities are reused when attached. Its seed 43 collapsed,
so it is retrained here.

**How it works.** Every member of the full-data ensemble is trained with the same flags
and seed on the 85% training split, then scored on the fixed holdout. The members are
averaged exactly as the final fit averages them, and the final combination is compared
with the current best by paired bootstrap. That makes this the evidence for or against
submitting the full-data version.

**Reuse.** A member whose identical recipe and seed already ran on the holdout is reused
when that run's output is attached. Everything else is trained here. The `MEMBERS`
list in the next cell mirrors the full-data fit; if you change the final fit's flags,
change them here too.

**Time:** about 8 h standalone (TAPT, then six 3-epoch fits two at a time), about 6.5 h with Run 28 attached.

**Settings:** GPU T4 x2, Internet on. No token and no attached input are needed.
Save & Run All. A hard stop at 11 h means the notebook always finishes and packages
what it has.

In [ ]:
import json, os, pathlib, subprocess, sys, time
from fnmatch import fnmatch
T_START = time.time()
HARD_STOP = T_START + 11 * 3600   # Kaggle kills at 12 h and then saves nothing

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/tmp/hastika"             # outside /kaggle/working: the output holds results only


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. Turn on Settings -> Internet, then restart.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                              # Gemma-4 is not gated; no token is needed

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
OUT = pathlib.Path("/kaggle/working/a35_outputs")
RUNS, LOGS, SUB = OUT / "runs", OUT / "logs", OUT / "submissions"
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap

TASK = "a"
CLASSES = ["Non-Hate", "Hate"]
LABEL_COL = "Label"
SPLIT = pathlib.Path(WORK) / "data/derived/task_a_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(df), len(holdout)) == (7193, 1079), (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "815110ff24", HOLDOUT_FP
y_ho = holdout[LABEL_COL].map(CLASSES.index).to_numpy()
TEST_CSV = pathlib.Path("/tmp/test_inputs.csv")
pd.read_csv(pathlib.Path(WORK) / "data/raw/hastika_binary_test.csv")[["id", "Comment"]].to_csv(TEST_CSV, index=False)
test = pd.read_csv(TEST_CSV)
EXTERNAL = pathlib.Path(WORK) / "data/external/offenseval_kn.csv"
print(f"task {TASK.upper()}: train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), "
      f"all {len(df)}, test {len(test)}")


def report(y, probs, name):
    pred = probs.argmax(1)
    per = f1_score(y, pred, average=None, labels=range(len(CLASSES)), zero_division=0)
    macro = float(f1_score(y, pred, average="macro"))
    print(f"{name:34s} macro-F1 {macro:.4f}  acc {accuracy_score(y, pred):.4f}  | "
          + "  ".join(f"{c[:5]} {f:.3f}" for c, f in zip(CLASSES, per)))
    return macro


def boot_line(label, a, b):
    r = paired_bootstrap(y_ho, a, b)
    print(f"{label:28s} {r['diff']:+.4f}  95% CI [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}]  "
          f"P(better) {r['p_better']:.2f}")
    return r


# Earlier runs' outputs are reused when attached (Add Input -> Notebook Output), found by
# walking every input with symlinks followed. Nothing below *requires* them.
def find_prior(pattern):
    hits = []
    for root, _, files in os.walk("/kaggle/input", followlinks=True):
        for f in files:
            p = os.path.join(root, f)
            if fnmatch(p, pattern):
                hits.append(pathlib.Path(p))
    return sorted(hits)


def show_inputs():
    if not os.path.isdir("/kaggle/input"):
        print("no /kaggle/input")
        return
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 4:
            print("  " * depth + (os.path.basename(root) or "/kaggle/input") + f"/  ({len(files)} files)")


print("attached inputs:")
show_inputs()

In [ ]:
# LLM stack in its own environment (transformers >= 5.12 for Gemma-4); Kaggle's torch and
# CUDA are reused via --system-site-packages. The repo's MuRIL/SVM code stays on the
# system environment.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import transformers, peft, bitsandbytes; print('LLM env: "
                "transformers', transformers.__version__, '| peft', peft.__version__)"], check=True)

MODEL = "google/gemma-4-12B"
S = "gemma-4-12b"


def download(model=MODEL):
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])


def llm_job(name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def tapt_job(name, texts):
    """Gemma TAPT: LoRA next-token training on unlabelled comments (the MuRIL +2.9 lesson)."""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_tapt --model {MODEL} "
                    f"--texts {' '.join(map(str, texts))} --out {RUNS / name} --cache {HF_CACHE}")}


def load_probs(run_dir, csv):
    """<input stem>_probs.npy, or None when it is missing, belongs to other rows (another
    task's output found among the inputs), or is collapsed onto one class (Task A Run 28's
    seed 43 predicted Non-Hate for every row)."""
    run_dir, csv = pathlib.Path(run_dir), pathlib.Path(csv)
    p = run_dir / f"{csv.stem}_probs.npy"
    if not p.exists():
        return None
    ids = pd.read_csv(run_dir / f"{csv.stem}_ids.csv")["id"].tolist()
    if ids != pd.read_csv(csv)["id"].tolist():
        return None
    probs = np.load(p)
    share = np.bincount(probs.argmax(1), minlength=probs.shape[1]) / len(probs)
    if share.max() > 0.9:
        print(f"skipping {run_dir.name}: collapsed, {share.max():.0%} of rows in one class")
        return None
    return probs


def run_jobs(jobs, on_done=None):
    """run_queue with two automatic retries per job: exit code 4 (collapsed after epoch 1)
    reruns with seed + 1000; out of GPU memory reruns at micro-batch 4 (same effective
    batch). on_done sees each job's final outcome."""
    import re
    registry, retried = {j["name"]: j for j in jobs}, set()

    def done(name, code):
        job, nxt = registry.get(name), []
        if code and job is not None and name not in retried:
            log = open(job["log"], errors="replace").read()
            seed = re.search(r"--seed (\d+)", job["cmd"])
            if code == 4 and seed:
                print(f"{name} collapsed -- rerunning with seed {int(seed.group(1)) + 1000}")
                nxt = [{**job, "cmd": job["cmd"] + f" --seed {int(seed.group(1)) + 1000}"}]
            elif "OutOfMemoryError" in log and "llm_tapt" in job["cmd"]:
                # TAPT already runs at micro-batch 1 by default: last resort is shorter text
                print(f"{name} ran out of GPU memory -- rerunning TAPT at 64 tokens")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 1 --grad-accum 16 --max-len 64 --logit-chunk 8"}]
            elif "OutOfMemoryError" in log:
                print(f"{name} ran out of GPU memory -- rerunning at micro-batch 4")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"}]
            if nxt:
                retried.add(name)
        if not nxt and on_done is not None:
            nxt = list(on_done(name, code) or [])
        for j in nxt:
            registry[j["name"]] = j
        return nxt

    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=done, stop_at=HARD_STOP)


def write_zip(name, probs, recommended=False):
    import zipfile
    assert probs.shape == (len(test), len(CLASSES)), probs.shape
    pred = pd.DataFrame({"id": test["id"], "label": [CLASSES[i] for i in probs.argmax(1)]})
    assert pred["id"].is_unique and set(pred["label"]) <= set(CLASSES)
    np.save(SUB / f"{name}_test_probs.npy", probs)
    csv = SUB / f"predictions_{name}.csv"
    pred.to_csv(csv, index=False)
    path = SUB / f"{'RECOMMENDED_' if recommended else ''}{name}.zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv, "predictions.csv")
    prior = df[LABEL_COL].value_counts(normalize=True)
    counts = pred["label"].value_counts(normalize=True)
    print(f"{path.name}  predicted/prior: "
          + "  ".join(f"{c[:5]} {counts.get(c, 0):.1%}/{prior[c]:.1%}" for c in CLASSES))
    return path


download()

In [ ]:
# Each member: (tag, seed, classifier flags, holdout files of earlier runs that are the
# same recipe and seed -- reused when attached). Edit to mirror the full-data fit exactly.
MEMBERS = [
    ('tapt', 42, '{tapt}', []),
    ('tapt', 43, '{tapt}', []),
    ('tapt', 44, '{tapt}', []),
    ('tapt', 45, '{tapt}', []),
    ('base', 42, '', ['*/runs/ho_s{seed}/holdout_probs.npy', '*/runs/base_s{seed}/holdout_probs.npy']),
    ('base', 43, '', ['*/runs/ho_s{seed}/holdout_probs.npy', '*/runs/base_s{seed}/holdout_probs.npy']),
]
GROUPS = {
    'current best: Run 28 recipe x2': ['base_s42', 'base_s43'],
    'TAPT x2': ['tapt_s42', 'tapt_s43'],
    'FINAL: TAPT x4 (= Run 33)': ['tapt_s42', 'tapt_s43', 'tapt_s44', 'tapt_s45'],
}            # group name -> member keys "<tag>_s<seed>"
FINAL, REFERENCE = 'FINAL: TAPT x4 (= Run 33)', 'current best: Run 28 recipe x2'
VERDICT = ('TAPT x2', 'current best: Run 28 recipe x2')          # (candidate, baseline) groups that decide the verdict
HO = dict(train=SPLIT / "train.csv", predict=[SPLIT / "holdout.csv"], evalf=SPLIT / "holdout.csv")

probs, jobs, after_tapt = {}, [], []
TAPT_DIR = RUNS / "tapt_train"
for tag, seed, extra, reuse in MEMBERS:
    key = f"{tag}_s{seed}"
    for pat in reuse:
        got = [load_probs(h.parent, SPLIT / "holdout.csv") for h in find_prior(pat.format(seed=seed))]
        got = [g for g in got if g is not None]
        if got:
            probs[key] = got[0]
            print(f"{key}: reused from an attached run ({pat.format(seed=seed)})")
            break
    else:
        if "{tapt}" in extra:      # a TAPT member: classifier continues from the TAPT adapter
            after_tapt.append(llm_job(key, seed=seed, extra=extra.format(tapt=f"--init-adapter {TAPT_DIR}"), **HO))
        else:
            jobs.append(llm_job(key, seed=seed, extra=extra, **HO))
state = {"tapt_ok": not after_tapt}
if after_tapt:   # TAPT on training-split text + external corpus only, never holdout or test
    jobs.insert(0, tapt_job("tapt_train", [SPLIT / "train.csv", EXTERNAL]))


def on_done(name, code):
    if name == "tapt_train":
        state["tapt_ok"] = code == 0
        if code == 0:
            return after_tapt
        print(f"TAPT FAILED (exit {code}) -- no TAPT member is trained; see {LOGS / 'tapt_train.log'}")
    return []


print("training on the 85% split:", [j["name"] for j in jobs + after_tapt])
codes = run_jobs(jobs, on_done)
for j in jobs + after_tapt:
    p = load_probs(RUNS / j["name"], SPLIT / "holdout.csv")
    if p is not None:
        probs[j["name"]] = p
print("exit codes:", codes, f"| {(time.time() - T_START) / 3600:.1f} h")

In [ ]:
for key in sorted(probs):
    report(y_ho, probs[key], key)
print()
avg, scores = {}, {}
for name, keys in GROUPS.items():
    have = [k for k in keys if k in probs]
    if have:
        avg[name] = np.mean([probs[k] for k in have], 0)
        scores[name] = report(y_ho, avg[name], f"{name} ({len(have)}/{len(keys)})")
print()
boots = {}
pairs = [(FINAL, o) for o in avg if o != FINAL] + ([VERDICT] if VERDICT[0] != FINAL else [])
for a, b_ in pairs:
    if a in avg and b_ in avg:
        boots[f"{a} - {b_}"] = boot_line(f"{a[:13]} - {b_[:13]}", avg[a], avg[b_])
if f"{VERDICT[0]} - {VERDICT[1]}" in boots:
    b = boots[f"{VERDICT[0]} - {VERDICT[1]}"]
    verdict = (f"{VERDICT[0]} BEATS {VERDICT[1]}" if b["ci95"][0] > 0 else
               f"{VERDICT[0]} is probably better" if b["p_better"] >= 0.7 else
               "no clear gain over the current best" if b["p_better"] >= 0.3 else
               "probably WORSE -- submit the current best")
    print(f"\nverdict ({VERDICT[0]} vs {VERDICT[1]}): {verdict} (P(better) {b['p_better']:.2f})")
if not state["tapt_ok"]:
    raise SystemExit("TAPT did not complete -- no TAPT result. See the tapt_train log above.")
json.dump({"commit": head, "holdout_fingerprint": HOLDOUT_FP, "scores": scores,
           "bootstrap": boots, "members": sorted(probs), "exit_codes": codes},
          open(OUT / "result.json", "w"), indent=2)